# Fire / Smoke / Neutral Classification — ResNet50 Transfer Learning (PyTorch)

Dataset: [fire-and-smoke-dataset](https://www.kaggle.com/datasets/hafilrazak/fire-and-smoke-dataset)
(structure: `FIRE-SMOKE-DATASET/Train/<class>/...` and `FIRE-SMOKE-DATASET/Test/<class>/...`)

This notebook:
1. Loads the dataset's existing Train/Test split directly from `/kaggle/input`
2. Builds a ResNet50 transfer-learning model (ImageNet weights, torchvision)
3. Trains the classification head, then fine-tunes the top of ResNet50
4. Evaluates on the dataset's own Test folder with a confusion matrix + classification report
5. Saves the trained model as a **`.pth`** state dict

**Before running:** add the dataset to this notebook (File → Add Input → search `fire-and-smoke-dataset`), and turn on a GPU accelerator + Internet access (Settings → Accelerator / Internet).

In [ ]:
import os, json, pathlib, random, copy, time

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import ResNet50_Weights
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Locate the Train / Test folders

This dataset ships pre-split into `Train` and `Test` folders (each containing one
subfolder per class), rather than one flat folder of classes. The cell below
searches under `/kaggle/input` for that `Train`/`Test` pair so it doesn't matter
exactly how deeply nested they are (e.g. `fire-and-smoke-dataset/FIRE-SMOKE-DATASET/Train`).

In [ ]:
INPUT_ROOT = pathlib.Path("/kaggle/input")

def find_train_test_dirs(root):
    train_dir, test_dir = None, None
    for p in root.rglob("*"):
        if not p.is_dir():
            continue
        name = p.name.strip().lower()
        if name == "train" and train_dir is None:
            train_dir = p
        elif name in ("test", "val", "validation") and test_dir is None:
            test_dir = p
    return train_dir, test_dir

TRAIN_DIR, TEST_DIR = find_train_test_dirs(INPUT_ROOT)

if TRAIN_DIR is None or TEST_DIR is None:
    print("Available top-level input folders:", os.listdir(INPUT_ROOT))
    raise FileNotFoundError(
        "Couldn't auto-locate Train/Test folders. Make sure the dataset is added "
        "via File -> Add Input, then check the printed folder list above and set "
        "TRAIN_DIR / TEST_DIR manually if the names differ."
    )

print("Train dir:", TRAIN_DIR)
print("Test dir:", TEST_DIR)
print("Classes in Train:", sorted(d.name for d in TRAIN_DIR.iterdir() if d.is_dir()))
print("Classes in Test:", sorted(d.name for d in TEST_DIR.iterdir() if d.is_dir()))

## 2. Config

In [ ]:
IMG_SIZE = 224           # ResNet50 default input size
BATCH_SIZE = 32
VAL_FRACTION = 0.15      # carved out of the Train folder (Test folder is left untouched)
EPOCHS_HEAD = 40          # training just the classification head
EPOCHS_FINETUNE = 60      # fine-tuning the top of ResNet50 (40 + 60 = 100 total)
UNFREEZE_FROM = "layer4"  # unfreeze this block onward (layer1 < layer2 < layer3 < layer4 < fc)
NUM_WORKERS = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

## 3. Transforms + datasets

Augmentation (flip / rotation / color jitter) is applied only to the training split.
Validation and test use a plain resize + normalize. `Train` is split into train/val;
`Test` is used exactly as provided by the dataset.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load Train folder once (no transform yet) to split into train/val indices
full_train_dataset = datasets.ImageFolder(TRAIN_DIR)
CLASS_NAMES = full_train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print("Class names:", CLASS_NAMES)

n_total = len(full_train_dataset)
n_val = int(n_total * VAL_FRACTION)
n_train = n_total - n_val

train_subset, val_subset = random_split(
    full_train_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
print(f"Train: {n_train}  Val: {n_val}")

# Wrap so each split applies its own transform
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label

train_ds = TransformSubset(train_subset, train_transform)
val_ds = TransformSubset(val_subset, eval_transform)

# Test dataset: loaded directly from the dataset's own Test folder
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
if test_dataset.classes != CLASS_NAMES:
    print("WARNING: Test folder class order differs from Train:", test_dataset.classes)
print(f"Test: {len(test_dataset)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# Peek at a few training samples
def denormalize(img_tensor):
    img = img_tensor.numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

images, labels = next(iter(train_loader))
plt.figure(figsize=(10, 10))
for i in range(min(9, images.shape[0])):
    plt.subplot(3, 3, i + 1)
    plt.imshow(denormalize(images[i]))
    plt.title(CLASS_NAMES[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Build the ResNet50 transfer-learning model

Stage 1 trains only the new classification head with the ResNet50 backbone frozen.

In [ ]:
def build_model(num_classes):
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, num_classes),
    )
    return model

model = build_model(NUM_CLASSES).to(device)
print(model.fc)

## 5. Training / evaluation loop helpers

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss, running_correct, total = 0.0, 0, 0
    torch.set_grad_enabled(is_train)
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        if is_train:
            optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        if is_train:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        total += inputs.size(0)

    return running_loss / total, running_correct / total


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs, patience=5):
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    for epoch in range(epochs):
        t0 = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        if scheduler is not None:
            scheduler.step(val_loss)

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} ({time.time()-t0:.1f}s) - "
              f"loss: {train_loss:.4f} acc: {train_acc:.4f} - "
              f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    return model, history

## 6. Stage 1 — train the classification head

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6)

model, history_head = train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    epochs=EPOCHS_HEAD, patience=8,
)

## 7. Stage 2 — fine-tune the top of ResNet50

Unfreeze `layer4` (and the head) and continue training with a much smaller
learning rate so the pretrained weights aren't destroyed.

In [ ]:
for name, param in model.named_parameters():
    if UNFREEZE_FROM in name or "fc" in name:
        param.requires_grad = True

trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable parameters: {sum(p.numel() for p in trainable_params):,}")

optimizer_ft = optim.Adam(trainable_params, lr=1e-5)
scheduler_ft = optim.lr_scheduler.ReduceLROnPlateau(optimizer_ft, mode="min", factor=0.5, patience=3, min_lr=1e-7)

model, history_finetune = train_model(
    model, train_loader, val_loader, criterion, optimizer_ft, scheduler_ft,
    epochs=EPOCHS_FINETUNE, patience=10,
)

## 8. Training curves

In [ ]:
def combine(key):
    return history_head[key] + history_finetune[key]

acc, val_acc = combine("accuracy"), combine("val_accuracy")
loss, val_loss = combine("loss"), combine("val_loss")
switch_epoch = len(history_head["accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(acc, label="train"); axes[0].plot(val_acc, label="val")
axes[0].axvline(switch_epoch, color="gray", linestyle="--", label="fine-tune start")
axes[0].set_title("Accuracy"); axes[0].legend()

axes[1].plot(loss, label="train"); axes[1].plot(val_loss, label="val")
axes[1].axvline(switch_epoch, color="gray", linestyle="--", label="fine-tune start")
axes[1].set_title("Loss"); axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Evaluate on the dataset's Test folder

In [ ]:
test_loss, test_acc = run_epoch(model, test_loader, criterion, optimizer=None)
print(f"Test accuracy: {test_acc:.4f}   Test loss: {test_loss:.4f}")

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = outputs.argmax(1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()

## 10. Save the model as `.pth`

Saves the **state dict** (recommended PyTorch practice) plus the class names,
so the model can be reloaded with the same architecture later.

In [ ]:
SAVE_PATH = "fire_smoke_resnet50_final.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "img_size": IMG_SIZE,
    "mean": IMAGENET_MEAN,
    "std": IMAGENET_STD,
}, SAVE_PATH)

print("Saved model to", SAVE_PATH)

## 11. Reload + run inference on a single image

Demonstrates loading the `.pth` checkpoint from scratch (as you would in a
separate script/app) and predicting on one image.

In [ ]:
from PIL import Image

def load_model_for_inference(path):
    checkpoint = torch.load(path, map_location=device)
    m = build_model(len(checkpoint["class_names"]))
    m.load_state_dict(checkpoint["model_state_dict"])
    m.to(device)
    m.eval()
    return m, checkpoint["class_names"], checkpoint["img_size"], checkpoint["mean"], checkpoint["std"]

def predict_image(img_path, model, class_names, img_size, mean, std):
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    img = Image.open(img_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx]), dict(zip(class_names, probs.tolist()))

# Example:
# loaded_model, names, size, mean, std = load_model_for_inference(SAVE_PATH)
# label, confidence, all_probs = predict_image(str(TEST_DIR / "<class>" / "<some_image>.jpg"),
#                                               loaded_model, names, size, mean, std)
# print(label, confidence, all_probs)